# 02 — Activation Funnel

Event funnel + time-to-value. Week 1 primary notebook.

**Outputs:** `analysis/outputs/activation_funnel.png`

In [ ]:
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) if os.path.basename(os.getcwd()) == "analysis" else os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import asyncio
from analysis.lib.pa_charts import plot_funnel
from db import get_db, init_db
from services import AnalyticsService

OUTPUT_DIR = os.path.join(REPO_ROOT, "analysis", "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
async def load_funnel():
    db_path = os.path.join(REPO_ROOT, "studybuddy.db")
    if not os.path.exists(db_path):
        return {"event_steps": []}, {}
    db = await get_db(db_path)
    await init_db(db)
    analytics = AnalyticsService(db)
    funnel = await analytics.compute_funnel()
    activation = await analytics.compute_activation_metrics()
    await db.close()
    return funnel, activation

funnel, activation = asyncio.run(load_funnel())
print("Activation 24h:", activation.get("pct_first_session_within_24h"))
print("Time-to-value:", activation.get("time_to_hours", {}))

In [ ]:
steps = funnel.get("event_steps", [])
if steps:
    fig = plot_funnel(steps, title="Palph Activation Funnel (events)")
    out = os.path.join(OUTPUT_DIR, "activation_funnel.png")
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"Saved: {out}")
else:
    print("No funnel data yet.")

## Key findings

- Main drop-off step: ___
- Median hours to session_started: ___
- Recommendation: ___